# 🌾 Agricultural Outlook Scenario Explorer — World Wheat

**Methodology:** Extends the [OECD-FAO Agricultural Outlook 2023-2032](https://doi.org/10.1787/08801ab7-en) baseline for world wheat under three user-configurable supply/demand shocks.

**Units:** All quantities in million metric tonnes (Mt) unless noted.

---

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import plotly.io as pio
pio.renderers.default = 'notebook'

from data_loader import load_oecd_fao_wheat
from scenario_engine import build_all_scenarios, SCENARIOS, project_scenario, Scenario
from visualise import (
    supply_demand_balance,
    stock_use_comparison,
    trade_balance_comparison,
    price_outlook_comparison,
    scenario_dashboard,
)

## 1  Load data

In [ ]:
df_raw = load_oecd_fao_wheat()
print(f"Loaded {len(df_raw)} years: {df_raw['year'].min()}–{df_raw['year'].max()}")
df_raw[['year','production_kt','consumption_kt','closing_stocks_kt','stock_use_pct','world_price_usd_t']].tail(12)

## 2  Compute baseline + standard scenarios

In [ ]:
all_df = build_all_scenarios(df_raw)
print("Scenarios:", all_df['scenario'].unique())

# Summary at end of horizon
(
    all_df[all_df['year']==2032]
    [['scenario_label','production_kt','consumption_kt',
      'closing_stocks_kt','stock_use_pct','world_price_usd_t']]
    .assign(
        production_mt=lambda d: (d.production_kt/1000).round(1),
        consumption_mt=lambda d: (d.consumption_kt/1000).round(1),
        stocks_mt=lambda d: (d.closing_stocks_kt/1000).round(1),
    )
    [['scenario_label','production_mt','consumption_mt',
      'stocks_mt','stock_use_pct','world_price_usd_t']]
    .rename(columns=dict(
        scenario_label='Scenario', production_mt='Prod (Mt)',
        consumption_mt='Cons (Mt)', stocks_mt='Stocks (Mt)',
        stock_use_pct='S/U (%)', world_price_usd_t='Price ($/t)'
    ))
    .reset_index(drop=True)
)

## 3  Briefing dashboard (2×2 figure)

In [ ]:
fig_dash = scenario_dashboard(all_df)
fig_dash.show()

## 4  Individual charts

In [ ]:
# Supply & demand — baseline
supply_demand_balance(all_df, 'baseline').show()

In [ ]:
# Supply & demand — low yield scenario
supply_demand_balance(all_df, 'low_yield').show()

In [ ]:
stock_use_comparison(all_df).show()

In [ ]:
trade_balance_comparison(all_df).show()

In [ ]:
price_outlook_comparison(all_df).show()

## 5  Custom scenario — define your own shock

Change `yield_delta_pp` and `demand_delta_pp` below and re-run the cell.

In [ ]:
# ── USER-ADJUSTABLE PARAMETERS ──────────────────────────────────────────
YIELD_SHOCK_PP   = -2.0   # e.g. -2.0 = production grows 2 pp/yr slower than baseline
DEMAND_SHOCK_PP  =  1.0   # e.g. +1.0 = consumption grows 1 pp/yr faster than baseline
SCENARIO_NAME    = "my_shock"
SCENARIO_LABEL   = f"Custom: yield {YIELD_SHOCK_PP:+.1f} pp, demand {DEMAND_SHOCK_PP:+.1f} pp"
# ────────────────────────────────────────────────────────────────────────

custom_scenario = Scenario(
    name=SCENARIO_NAME,
    label=SCENARIO_LABEL,
    color='#7b2fbe',
    yield_delta_pp=YIELD_SHOCK_PP,
    demand_delta_pp=DEMAND_SHOCK_PP,
)

df_custom = project_scenario(df_raw, custom_scenario)

# Add to the main dataframe for combined plotting
import pandas as pd
combined = pd.concat([all_df, df_custom], ignore_index=True)

# Show stock-to-use with custom scenario highlighted
stock_use_comparison(
    combined,
    scenarios=['baseline', 'low_yield', 'high_yield', SCENARIO_NAME]
).show()

## 6  Export figures to disk

In [ ]:
from visualise import save_all_figures
os.makedirs('../figures', exist_ok=True)
saved = save_all_figures(all_df, output_dir='../figures', formats=['html'])
print(f'Saved {len(saved)} HTML figures to ../figures/')